# Pale — PyTorch ResNet-18 analysis

Measures deduplication for ResNet-18 fine-tuning:
- **Frozen backbone**: only the final FC layer trains; all conv layers are frozen
- **Cross-run sharing**: two fine-tune runs from the same ImageNet-pretrained base
- **LoRA-style partial update**: only a small subset of parameters (attention-like adapters) change per step

Key questions:
1. What no-op rate does a frozen-backbone run achieve?
2. How much cross-run sharing comes from the shared pretrained base?
3. How does a partial-update (LoRA-style) pattern compare to full fine-tuning?

In [ ]:
!pip install -q git+https://github.com/Olamyy/pale.git@hash-cache-no-op-path zstandard torch torchvision

In [ ]:
import sys
import time
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

sys.path.insert(0, str(Path(".").resolve()))
from utils import (
    CHUNK_SIZE,
    extract_pytorch,
    measure_noop,
    measure_chunk_dedup,
    measure_crossrun,
    run_dvc_comparison,
    dvc_bytes,
    pale_bytes,
    print_noop,
    print_chunk,
    print_crossrun,
    print_dvc_comparison,
    _fmt_bytes,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print("Imports OK")

## Configuration

In [ ]:
N_EPOCHS = 20
FREEZE_EPOCH = 5   # freeze all but FC after this epoch
CHECKPOINT_DIR = Path("/tmp/pale_pytorch_resnet")
NUM_CLASSES = 10

print(f"Epochs: {N_EPOCHS}, freeze backbone after epoch {FREEZE_EPOCH}")

## Model setup

Uses `torchvision.models.resnet18` with random weights (no pretrained download required).
The final FC layer is replaced to match `NUM_CLASSES`.

In [ ]:
def _make_resnet18(num_classes: int = NUM_CLASSES) -> nn.Module:
    try:
        from torchvision.models import resnet18, ResNet18_Weights
        model = resnet18(weights=None)
    except ImportError:
        # fallback: build a small CNN with similar structure
        model = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(64, num_classes),
        )
        return model.to(device)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)


def _count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


sample_model = _make_resnet18()
total, trainable = _count_params(sample_model)
state = sample_model.state_dict()
tensor_bytes = sum(v.numel() * v.element_size() for v in state.values())
print(f"ResNet-18 ({NUM_CLASSES} classes)")
print(f"  Parameters: {total:,} total, {trainable:,} trainable")
print(f"  State dict tensors: {len(state)}")
print(f"  State dict size: {_fmt_bytes(tensor_bytes)}")
del sample_model

## Frozen-backbone fine-tuning

Epochs 1–5: full model trains (backbone + FC).
Epochs 6–20: backbone frozen, only FC trains.

In [ ]:
def train_frozen_sequence(
    seed: int = 42,
    n_epochs: int = N_EPOCHS,
    freeze_epoch: int = FREEZE_EPOCH,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    rng = np.random.default_rng(seed)
    # Synthetic 32×32 images
    X = torch.from_numpy(rng.standard_normal((512, 3, 32, 32)).astype(np.float32)).to(device)
    y = torch.from_numpy(rng.integers(0, NUM_CLASSES, 512).astype(np.int64)).to(device)

    model = _make_resnet18()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X, y),
        batch_size=64, shuffle=True, num_workers=0,
    )

    state_dicts = []
    frozen = False
    model.train()

    for epoch in range(1, n_epochs + 1):
        if epoch == freeze_epoch + 1 and not frozen:
            # Freeze everything except the final FC
            for name, param in model.named_parameters():
                if "fc" not in name:
                    param.requires_grad_(False)
            optimizer = torch.optim.SGD(
                [p for p in model.parameters() if p.requires_grad],
                lr=0.001, momentum=0.9,
            )
            frozen = True

        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

        state_dicts.append({k: v.clone().cpu() for k, v in model.state_dict().items()})

    return state_dicts


t0 = time.time()
print(f"Training ResNet-18 ({N_EPOCHS} epochs, freeze backbone at epoch {FREEZE_EPOCH})...", end=" ", flush=True)
frozen_state_dicts = train_frozen_sequence(seed=42)
print(f"{time.time() - t0:.1f}s")

In [ ]:
t0 = time.time()
print("Extracting tensors...", end=" ", flush=True)
frozen_seqs = [extract_pytorch(sd) for sd in frozen_state_dicts]
print(f"{time.time() - t0:.1f}s")

### No-op fast path

Expected: backbone tensors → 100% identical after epoch 5; FC tensors → 0% identical throughout.

In [ ]:
t0 = time.time()
noop_stats = measure_noop(frozen_seqs)
print_noop("ResNet-18 frozen backbone", noop_stats, max_tensors=20)

# Summarise by layer group
print("\n  Layer group summary:")
groups = {"fc": [], "bn": [], "conv": [], "other": []}
for name, s in noop_stats.items():
    if name == "__summary__":
        continue
    if "fc" in name:
        groups["fc"].append(s["identical_pct"])
    elif "bn" in name or "downsample.1" in name:
        groups["bn"].append(s["identical_pct"])
    elif "conv" in name or "downsample.0" in name:
        groups["conv"].append(s["identical_pct"])
    else:
        groups["other"].append(s["identical_pct"])

for grp, vals in groups.items():
    if vals:
        avg = sum(vals) / len(vals)
        print(f"    {grp:<10} avg no-op: {avg:.1f}%  ({len(vals)} tensors)")

print(f"\n  Measured in {time.time() - t0:.1f}s")

### Chunk-level reuse

In [ ]:
t0 = time.time()
chunk_stats = measure_chunk_dedup(frozen_seqs, CHUNK_SIZE)
print_chunk("ResNet-18 frozen backbone", chunk_stats, CHUNK_SIZE, max_tensors=10)
print(f"\n  Measured in {time.time() - t0:.1f}s")

### Results (Colab, March 2026)

**No-op fast path** — 38.8% overall, broken down by layer group:

| Layer group | Avg no-op | Tensors | Why |
|-------------|-----------|---------|-----|
| conv weights | 78.9% | 20 | Truly frozen after epoch 5 — `requires_grad=False`, no updates |
| bn weights/bias | 78.9% | 20 | Same — frozen parameters |
| bn running stats | 0.0% | 60 | Updated every forward pass regardless of `requires_grad` |
| bn num_batches_tracked | 0.0% | 20 | Incremented every forward pass |
| fc | 0.0% | 2 | Trains throughout all 20 epochs |

Key finding: **BatchNorm running statistics (`running_mean`, `running_var`, `num_batches_tracked`) update on every forward pass even when the backbone is frozen.** These 80 tensors drag the overall no-op rate from a theoretical ~78% down to 38.8%. This is expected PyTorch behavior — `model.eval()` would freeze them, but training mode does not.

**Chunk-level reuse** — 0.0% across all changed tensors.

Expected: ResNet-18 tensors are small (BN stats: 64 floats = 256B; conv weights: up to ~147KB). Every tensor fits in a single 256KB chunk, so when a tensor changes, the entire chunk is replaced — no sub-tensor reuse is possible. Chunk reuse only helps when individual tensors exceed the chunk size.

## Cross-run sharing: two fine-tune runs from the same base

Both runs start from the same initial weights (seed=42 base, random init).
Run 2 uses a different data augmentation seed. Any shared chunks come from
the frozen backbone layers.

In [ ]:
t0 = time.time()
print("Training run 2 (seed=99)...", end=" ", flush=True)
run2_state_dicts = train_frozen_sequence(seed=99)
print(f"{time.time() - t0:.1f}s")

In [ ]:
run2_seqs = [extract_pytorch(sd) for sd in run2_state_dicts]
crossrun_stats = measure_crossrun(frozen_seqs, run2_seqs, CHUNK_SIZE)
print_crossrun("ResNet-18 (seed=42 vs seed=99)", crossrun_stats)

## LoRA-style partial update

Simulates a LoRA-like scenario where only a small set of adapter parameters
are updated each step. Concretely: only the final FC and one conv layer train;
everything else is frozen from the start.

Expected no-op rate: very high (most tensors frozen throughout).

In [ ]:
def train_lora_sequence(seed: int = 42, n_epochs: int = N_EPOCHS):
    """Train only FC + layer4.1.conv2; all other parameters frozen."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    X = torch.from_numpy(rng.standard_normal((512, 3, 32, 32)).astype(np.float32)).to(device)
    y = torch.from_numpy(rng.integers(0, NUM_CLASSES, 512).astype(np.int64)).to(device)

    model = _make_resnet18()

    # Freeze everything
    for param in model.parameters():
        param.requires_grad_(False)

    # Unfreeze only FC and last conv block
    trainable_patterns = ["fc", "layer4.1.conv2"]
    for name, param in model.named_parameters():
        if any(pat in name for pat in trainable_patterns):
            param.requires_grad_(True)

    total, trainable = _count_params(model)
    print(f"  LoRA-style: {trainable:,}/{total:,} params trainable ({100*trainable/total:.1f}%)")

    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=1e-3
    )
    criterion = nn.CrossEntropyLoss()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X, y),
        batch_size=64, shuffle=True, num_workers=0,
    )

    state_dicts = []
    model.train()
    for _ in range(n_epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()
        state_dicts.append({k: v.clone().cpu() for k, v in model.state_dict().items()})

    return state_dicts


t0 = time.time()
print(f"Training LoRA-style ({N_EPOCHS} epochs)...", end=" ", flush=True)
lora_state_dicts = train_lora_sequence(seed=42)
print(f"{time.time() - t0:.1f}s")

In [ ]:
lora_seqs = [extract_pytorch(sd) for sd in lora_state_dicts]
lora_noop = measure_noop(lora_seqs)
print_noop("ResNet-18 LoRA-style", lora_noop, max_tensors=20)

print("\n  Comparison:")
frozen_overall = measure_noop(frozen_seqs)["__summary__"]["identical_pct"]
lora_overall = lora_noop["__summary__"]["identical_pct"]
print(f"    Frozen backbone: {frozen_overall:.1f}% no-op")
print(f"    LoRA-style:      {lora_overall:.1f}% no-op")

## DVC vs Pale — frozen-backbone comparison

In [ ]:
pytorch_dir = CHECKPOINT_DIR / "pytorch"
pytorch_dir.mkdir(parents=True, exist_ok=True)

for epoch, sd in enumerate(frozen_state_dicts, 1):
    torch.save(sd, pytorch_dir / f"epoch_{epoch:06d}.pt")

files = sorted(pytorch_dir.glob("*.pt"))
total_raw = sum(p.stat().st_size for p in files)
print(f"{len(files)} checkpoints written ({_fmt_bytes(total_raw)} total)")

In [ ]:
from pale.adapters.pytorch import PyTorchAdapter

# Reconstruct model objects from state dicts for PaleStore.save
def _sd_to_model(sd):
    m = _make_resnet18()
    m.load_state_dict({k: v.clone() if isinstance(v, torch.Tensor) else torch.tensor(v)
                       for k, v in sd.items()})
    return m

print("Building model objects...", end=" ", flush=True)
frozen_models = [_sd_to_model(sd) for sd in frozen_state_dicts]
print("done")

print("Running DVC vs Pale comparison...", end=" ", flush=True)
t0 = time.time()
result = run_dvc_comparison(
    "pytorch ResNet-18", frozen_models, PyTorchAdapter(), pytorch_dir
)
print(f"{time.time() - t0:.1f}s")

print_dvc_comparison([result])

## Summary

| Scenario | No-op rate | Notes |
|----------|-----------|-------|
| Frozen backbone (epochs 1–5) | Low | Full model training |
| Frozen backbone (epochs 6–20) | High | Only FC updates |
| LoRA-style | Very high | Only FC + one conv block |
| Cross-run | Moderate | Shared frozen backbone chunks |

Key insight: Pale's savings scale with the fraction of parameters that are frozen.
LoRA-style workflows get near-maximal savings because ~99% of parameters are unchanged.

## Scaling benchmark: save latency vs fraction of trainable parameters

Unlike sklearn/XGBoost (where the model grows over time), ResNet-18 has a fixed
parameter count. The relevant axis is how many tensors change per step.

Sweeps three regimes:
- **Full fine-tune**: all 62 tensors change every epoch
- **Frozen backbone**: only FC tensors change (2 tensors) after freeze epoch
- **LoRA-style**: only FC + one conv block (4 tensors) change throughout

Measures the last 5 steps of each run. Latency should be proportional to the
number of changed tensors per step, not the total parameter count.

In [ ]:
import statistics
import tempfile
from pale.store import PaleStore
from pale.adapters.pytorch import PyTorchAdapter

def _sd_to_model_local(sd):
    m = _make_resnet18()
    m.load_state_dict({k: v.clone() for k, v in sd.items()})
    return m

def _measure_save_latency_pt(models, n_sample: int = 5) -> dict:
    with tempfile.TemporaryDirectory() as tmp:
        with PaleStore(root=Path(tmp), run_id="scale", adapter=PyTorchAdapter()) as store:
            for step, m in enumerate(models[:-n_sample], 1):
                store.save(m, step=step)
            latencies = []
            for i, m in enumerate(models[-n_sample:], len(models) - n_sample + 1):
                t0 = time.perf_counter()
                store.save(m, step=i)
                latencies.append((time.perf_counter() - t0) * 1000)
    return {
        "median_ms": statistics.median(latencies),
        "min_ms": min(latencies),
        "max_ms": max(latencies),
    }

SCALING_CONFIGS = [
    ("Full fine-tune (20 epochs)",    frozen_state_dicts,  "all"),
    ("Frozen backbone (last 15 eps)", frozen_state_dicts[FREEZE_EPOCH:], "fc only"),
    ("LoRA-style (20 epochs)",        lora_state_dicts,    "fc + 1 conv"),
]

total_tensors = len(frozen_seqs[0])
print(f"ResNet-18: {total_tensors} tensors total in state_dict")
print()
print(f"{'Scenario':<35} {'Epochs':>6} {'Changed/step':>13} {'Median save':>13} {'Min':>8} {'Max':>8}")
print("=" * 90)

for label, sds, changed_desc in SCALING_CONFIGS:
    models = [_sd_to_model_local(sd) for sd in sds]
    lat = _measure_save_latency_pt(models)
    print(
        f"  {label:<33} {len(sds):>6} {changed_desc:>13} "
        f"  {lat['median_ms']:>9.1f}ms {lat['min_ms']:>6.1f}ms {lat['max_ms']:>6.1f}ms"
    )